In [1]:
import gpt as g
import sys, os
import numpy as np

from scipy.linalg import expm
import time
import matplotlib.pyplot as plt
from gpt.qcd.gauge.smear import local_stout  
from gpt.ad import reverse as rad
from gpt.qcd.gauge.smear.differentiable import dft_diffeomorphism
import time

SharedMemoryNone: SharedMemoryAllocate 1073741824 GPU implementation 
0SharedMemoryNone:  SharedMemoryNone.cc acceleratorAllocDevice 1073741824bytes at 0x300000000 for comms buffers 

__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|_ |  |  |  |  |  |  |  |  |  |  |  | _|__
__|_                                    _|__
__|_   GGGG    RRRR    III    DDDD      _|__
__|_  G        R   R    I     D   D     _|__
__|_  G        R   R    I     D    D    _|__
__|_  G  GG    RRRR     I     D    D    _|__
__|_  G   G    R  R     I     D   D     _|__
__|_   GGGG    R   R   III    DDDD      _|__
__|_                                    _|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
  |  |  |  |  |  |  |  |  |  |  |  |  |  |  


Copyright (C) 2015 Peter Boyle, Azusa Yamaguchi, Guido Cossu, Antonin Portelli and other authors

This program is free software; you can redistribute it and/or modify
it under the term

In [2]:
def trace_U(U):                                                      
    return sum(v for v in sum(u[:].real for u in g.eval(g.trace(U)))) / (4 * size**4 ) / 3.

In [3]:
num_steps = 1
def ftg(U, eps):
    global num_steps

    ##### dmuAmu ##############
    #B = U[0] -  g.adj(g.cshift(U[0], 0, -1))
    B = U[0] -  g.cshift(U[0], 0, -1)
    for mu in [1,]:
        #B += U[mu] -  g.adj(g.cshift(U[mu], mu, -1))
        B += U[mu] -  g.cshift(U[mu], mu, -1)

   #### masks for all even/odd sites ########
    """
    grid_cb = grid.checkerboarded(g.redblack)
    one_cb = g.complex(grid_cb)
    one_cb[:] = 1

    masks = {}
    for p in [g.even, g.odd]:
        m = g.complex(grid)
        m[:] = 0
        one_cb.checkerboard(p)
        g.set_checkerboard(m, one_cb)
        masks[p] = m
    
    if num_steps // 2 == 0:
        mask, imask = masks[g.odd], masks[g.odd.inv()] 
    else:
        mask, imask = masks[g.even], masks[g.even.inv()]
    
    num_steps += 1
    fm = g(mask + 1e-15 * imask)
    #fm = mask + 1e-15 * imask
    
    ###### apply masks ########
    #B *= fm # causes issues with action log det routine
    #B = g(B*fm)
    """
    
    # apply gtf 
    U_prime = []
    for mu in [0,1]:
        U_mu_prime = g(
                g.matrix.exp(  - eps * g.qcd.gauge.project.traceless_anti_hermitian(B) )  
                * U[mu] * g.matrix.exp(  + eps * g.qcd.gauge.project.traceless_anti_hermitian( g.cshift(B, mu, +1) ) ) 
        )
        U_prime.append(U_mu_prime)
    
    return U_prime


In [4]:
size = 4
grid = g.grid([size, size], g.double)
rng = g.random("t")

U = g.qcd.gauge.unit(grid)
rng.normal_element(U)


GPT :       0.405992 s : Initializing gpt.random(t,vectorized_ranlux24_389_64) took 0.000126839 s


[lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double)]

In [5]:
def ft0(U):
    return ftg(U, eps=-5e-2)
    
#dft = dft_diffeomorphism(U, ft0)
fr = g.algorithms.optimize.fletcher_reeves
ls2 = g.algorithms.optimize.line_search_quadratic

dft = g.qcd.gauge.smear.differentiable_field_transformation(
    U,
    ft0,
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=100, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=100, restartlen=10),
    g.algorithms.optimize.non_linear_cg(
        maxiter=100, eps=1e-15, step=1e-1, line_search=ls2, beta=fr
    ),
)

dfm = dft.diffeomorphism()
ald = dft.action_log_det_jacobian()

In [6]:
# FT

W = U

Vgf = dft.inverse(U)

Vgf2 = dft.inverse(Vgf)

# conjugate momenta
U_mom = g.group.cartesian(Vgf2)
#U_mom = g.group.cartesian(U)

action_gauge_mom = g.qcd.scalar.action.mass_term()
action_gauge = g.qcd.gauge.action.wilson(10.0)

metro = g.algorithms.markov.metropolis(rng)
inv = g.algorithms.inverter
sympl = g.algorithms.integrator.symplectic

log = sympl.log()

pure_gauge = True


a_log_det = dft.action_log_det_jacobian()

GPT :       0.558087 s : non_linear_cg: iteration 0: f(x) = 3.310981643896348e-02, |df|/sqrt(dof) = 5.660435e-02, beta = 0, step = 0.7945607843578245
GPT :       1.332202 s : non_linear_cg: iteration 10: f(x) = 2.281028429392470e-18, |df|/sqrt(dof) = 4.530341e-10, beta = 0.01820093246697598, step = 0.842102006816854
GPT :       1.882410 s : non_linear_cg: converged in 18 iterations: f(x) = 9.128215328541225e-27, |df|/sqrt(dof) = 7.845978e-16
GPT :       1.971708 s : non_linear_cg: iteration 0: f(x) = 2.438600722534080e-02, |df|/sqrt(dof) = 5.121611e-02, beta = 0, step = 0.7654175598054295
GPT :       2.745131 s : non_linear_cg: iteration 10: f(x) = 1.663609157281765e-18, |df|/sqrt(dof) = 4.247332e-10, beta = 0.01775196232546497, step = 0.8315728110047463
GPT :       3.290674 s : non_linear_cg: converged in 18 iterations: f(x) = 1.101253750419445e-26, |df|/sqrt(dof) = 8.160119e-16


In [7]:
print(trace_U(W), trace_U(Vgf), trace_U(Vgf2))

[0.01501172] [0.01663096] [0.01792894]


In [9]:
mom = [g.group.cartesian(v) for v in Vgf]
mom_prime = g.copy(mom)
rng.normal_element(mom_prime)

mom2 = dfm.jacobian(Vgf, U, mom_prime) # this appears to have fixed it???
ald = a_log_det(Vgf + mom2) # don't know if this is the correct log det!!!
g.message("Action log det jac:", ald)

GPT :      18.080393 s : fgcr: converged in 17 iterations;  computed squared residual 8.815872e-26 / 1.540093e-24;  true squared residual 8.816766e-26 / 1.540093e-24
GPT :      18.570366 s : fgcr: converged in 16 iterations;  computed squared residual 1.290231e-24 / 1.411144e-24;  true squared residual 1.290190e-24 / 1.411144e-24
GPT :      18.570990 s : Action log det jac: 141.11435623937533


In [10]:
mom_step2 = [g.group.cartesian(v) for v in Vgf2]
mom_step_prime = g.copy(mom_step2)
rng.normal_element(mom_step_prime)

mom_step2_2 = dfm.jacobian(Vgf2, Vgf, mom_step_prime) # this appears to have fixed it???
ald1 = a_log_det(Vgf2 + mom_step2_2) # don't know if this is the correct log det!!!
g.message("Action log det jac:", ald1)

GPT :      19.467252 s : fgcr: converged in 16 iterations;  computed squared residual 3.738266e-25 / 1.436934e-24;  true squared residual 3.738594e-25 / 1.436934e-24
GPT :      19.960301 s : fgcr: converged in 16 iterations;  computed squared residual 2.314494e-25 / 1.300253e-24;  true squared residual 2.316573e-25 / 1.300253e-24
GPT :      19.961428 s : Action log det jac: 130.02525400805905


In [11]:
def hamiltonian(draw):
    global mom2, mom_step2_2
    if draw:
        rng.normal_element(U_mom)

        Vgf = dfm(Vgf2)
        W = dfm(Vgf)
        
        mom = [g.group.cartesian(v) for v in Vgf]
        #mom = [g.group.cartesian(u) for u in U]
        mom_prime = g.copy(mom)
        rng.normal_element(mom_prime)


        mom2 = dfm.jacobian(Vgf, W, mom_prime) # this appears to have fixed it???
        ald = a_log_det(Vgf + mom2)
        

        mom_step2 = [g.group.cartesian(v) for v in Vgf2]
        mom_step_prime = g.copy(mom_step2)
        rng.normal_element(mom_step_prime)

        mom_step2_2 = dfm.jacobian(Vgf2, Vgf, mom_step_prime) # this appears to have fixed it???
        ald1 = a_log_det(Vgf2 + mom_step2_2)
        
        #s = action_gauge(U)

        
        s = action_gauge(Vgf2)

        g.message("Gluonic action", s)
        g.message("Log-det 1 action", ald)
        g.message("Log-det 2 action", ald1)
        
        h = s + action_gauge_mom(U_mom) + ald + ald1
        #h = s + action_gauge_mom(U_mom)

    else:

        Vgf = dfm(Vgf2)
        W = dfm(Vgf)
        
        ald = a_log_det(Vgf + mom2)
        ald1 = a_log_det(Vgf2 + mom_step2_2)
        s = action_gauge(Vgf2)
        #s = action_gauge(U)
        
        h = s + action_gauge_mom(U_mom) + ald + ald1
    return h, s


In [12]:
hamiltonian(True)

GPT :      25.008623 s : fgcr: converged in 17 iterations;  computed squared residual 8.916985e-26 / 1.447144e-24;  true squared residual 8.901157e-26 / 1.447144e-24
GPT :      25.527722 s : fgcr: converged in 17 iterations;  computed squared residual 7.053504e-26 / 1.289285e-24;  true squared residual 7.060468e-26 / 1.289285e-24
GPT :      26.422209 s : fgcr: converged in 16 iterations;  computed squared residual 3.372028e-25 / 1.458240e-24;  true squared residual 3.373264e-25 / 1.458240e-24
GPT :      26.914917 s : fgcr: converged in 16 iterations;  computed squared residual 2.747892e-25 / 1.301072e-24;  true squared residual 2.746449e-25 / 1.301072e-24
GPT :      26.916578 s : Gluonic action 161.6109447593405
GPT :      26.916811 s : Log-det 1 action 128.92854276884503
GPT :      26.916966 s : Log-det 2 action 130.10716305627759


(np.float64(574.2582086752487), np.float64(161.6109447593405))

In [13]:
def log_det_force1():
    #g.message("Compute log_det force")
    x = log(lambda: dfm.jacobian(Vgf2, Vgf, a_log_det.gradient(Vgf + mom2, Vgf)), "log det1")()
    #g.message("second level force complete")
    return x

def log_det_force2():
    #g.message("Compute log_det force")
    x = log(lambda: a_log_det.gradient(Vgf2 + mom_step2_2, Vgf2), "log det2")()
    #g.message("second level force complete")
    return x

def gauge_force():
    #g.message("Compute gauge force")
    x = log(lambda: action_gauge.gradient(Vgf2, Vgf2), "gauge")()
    return x


In [14]:
iq = sympl.update_q(
    Vgf2, log(lambda: action_gauge_mom.gradient(U_mom, U_mom), "gauge_mom")
)

ip_gauge = sympl.update_p(U_mom, gauge_force)

ip_log_det1 = sympl.update_p(U_mom, log_det_force1)
ip_log_det2 = sympl.update_p(U_mom, log_det_force2)

# integrator
mdint = sympl.leap_frog( 1, ip_log_det2, sympl.leap_frog(1, ip_log_det1, sympl.leap_frog(1, ip_gauge, iq)) )


g.message(f"Integration scheme:\n{mdint}")

tau = 1.0
nsteps = 10



GPT :      45.955933 s : Integration scheme:
                       : leap_frog(1, P, leap_frog(1, P, leap_frog(1, P, Q)))
                       :   P(0.5, 0)
                       :   P(0.5, 0)
                       :   P(0.5, 0)
                       :   Q(1.0, 0)
                       :   P(0.5, 0)
                       :   P(0.5, 0)
                       :   P(0.5, 0)


In [16]:
no_accept_reject = False

def hmc(tau):
    accrej = metro(Vgf2)
    #accrej = metro(U)
    g.message("After metro")

    h0, s0 = hamiltonian(True)
    
    #its0 = nsteps - 1
    nsteps = 10
    for i in range(nsteps):
        mdint(tau/nsteps)

    g.message("After mdint(tau)")
    h1, s1 = hamiltonian(False)
    g.message("After H(false)")
    
    if no_accept_reject:
        return [True, s1 - s0, h1 - h0]
    else:
        return [accrej(h1, h0), s1 - s0, h1 - h0]
   

In [17]:
hmc(1.)

GPT :      72.286796 s : After metro
GPT :      73.254724 s : fgcr: converged in 17 iterations;  computed squared residual 7.629614e-26 / 1.567471e-24;  true squared residual 7.624572e-26 / 1.567471e-24
GPT :      73.783755 s : fgcr: converged in 17 iterations;  computed squared residual 7.310592e-26 / 1.366639e-24;  true squared residual 7.308326e-26 / 1.366639e-24
GPT :      74.659003 s : fgcr: converged in 16 iterations;  computed squared residual 1.519725e-25 / 1.334758e-24;  true squared residual 1.519087e-25 / 1.334758e-24
GPT :      75.151317 s : fgcr: converged in 16 iterations;  computed squared residual 1.488033e-25 / 1.204206e-24;  true squared residual 1.487725e-25 / 1.204206e-24
GPT :      75.152292 s : Gluonic action 161.6109447593405
GPT :      75.152480 s : Log-det 1 action 136.6638783598412
GPT :      75.152624 s : Log-det 2 action 120.42062323300866
GPT :      75.998785 s : fgcr: converged in 16 iterations;  computed squared residual 1.519725e-25 / 1.334758e-24;  true

[False, np.float64(-68.38139816724248), np.float64(13.1168568353213)]

In [18]:
accept, total = 0, 0

plaq_measurements = []
trace2_measurements = []
trace1_measurements = []
trace_measurements = []

for it in range(20):
    #pure_gauge = it < 10
    no_accept_reject = it < 10
    g.message(pure_gauge, no_accept_reject)

    a, dS, dH = hmc(tau)
    accept += a
    total += 1


    #Uft = U
    #for s in reversed(sm):
    #    Uft = s(Uft)
        
    plaq = g.qcd.gauge.plaquette(Vgf2)
    plaq_measurements.append(plaq)
    trace2_measurements.append(trace_U(Vgf2))
    trace1_measurements.append(trace_U(Vgf))
    trace_measurements.append(trace_U(W))
    
    g.message(f"HMC plaq = {plaq}, dS = {dS}, dH = {dH}, acceptance = {accept/total}")
    g.message(f"Timing:\n{log.time}")
    g.message(f"TRAJECTORY NUM {it}")

GPT :     158.528620 s : True True
GPT :     158.529895 s : After metro
GPT :     159.483675 s : fgcr: converged in 17 iterations;  computed squared residual 7.826313e-26 / 1.474024e-24;  true squared residual 7.834986e-26 / 1.474024e-24
GPT :     159.996065 s : fgcr: converged in 17 iterations;  computed squared residual 8.782577e-26 / 1.329833e-24;  true squared residual 8.769270e-26 / 1.329833e-24
GPT :     160.866993 s : fgcr: converged in 16 iterations;  computed squared residual 1.565820e-25 / 1.597399e-24;  true squared residual 1.568581e-25 / 1.597399e-24
GPT :     161.355596 s : fgcr: converged in 16 iterations;  computed squared residual 1.197224e-25 / 1.432909e-24;  true squared residual 1.197308e-25 / 1.432909e-24
GPT :     161.356341 s : Gluonic action 161.6109447593405
GPT :     161.356523 s : Log-det 1 action 132.98334965457033
GPT :     161.356668 s : Log-det 2 action 143.2909451733848
GPT :     162.184679 s : fgcr: converged in 16 iterations;  computed squared residual

KeyboardInterrupt: 

In [ ]:
plt.plot(plaq_measurements)

In [ ]:
plt.plot(trace_measurements, color='r')
plt.plot(trace1_measurements, color='g')
plt.plot(trace2_measurements, color='blue')

In [ ]:
meas = np.array(plaq_measurements)

print(f"Mean plaq = {np.mean(meas)}, std = {np.std(meas)}")